In [0]:
from pyspark.sql import functions as F

oncology_pat_ids = (
    spark.sql(
        """
        SELECT DISTINCT
            pa.PAT_ID
        FROM datahub_dev_bronze.datahub_clarity.mv_dm_patient_access AS pa
        INNER JOIN opsanalytics_adb_workspace01.oncology.mapping_department AS dg
            ON pa.DEPARTMENT_ID = dg.DEPARTMENT_ID
        WHERE pa.PAT_ID IS NOT NULL
        """
    )
)

In [0]:
# 3. Oracle applies the filter before returning data to Databricks.
ooracle_jdbc_url = dbutils.secrets.get(
    scope="oao_secrets",
    key="ORACLE_JDBC_URL",
)

oracle_password = dbutils.secrets.get(
    scope="oao_secrets",
    key="OAO_PRODUCTION",
)

# Without partitioning options, this is one JDBC partition/connection.
demographics_df = (
    spark.read
    .format("jdbc")
    .option("url", oracle_jdbc_url)
    .option(
        "dbtable",
        "OAO_PRODUCTION.MV_PATIENT_SELECT_DEMOGRAPHICS",
    )
    .option("user", "OAO_PRODUCTION")
    .option("password", oracle_password)
    .option("driver", "oracle.jdbc.OracleDriver")
    .option("oracle.net.ssl_server_dn_match", "true")
    .option("fetchsize", "10000")
    .load()
)

In [0]:
filtered_demographics_df = demographics_df.join(
    F.broadcast(oncology_pat_ids),
    on="PAT_ID",
    how="left_semi",
)

In [0]:
# 4. Spark only receives the filtered result and writes Delta.
(
    filtered_demographics_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "opsanalytics_adb_workspace01.oncology.mapping_mv_patient_select_demographics"
    )
)